# FYS4480/9480, week 48: coupled cluster with singles and doubles

Companion notebook to the Thursday lectures of `week48.tex`. Plain `numpy`. Three routes to the CCSD equations are
compared for a random Hamiltonian and random amplitudes:

1. **brute force**: $e^{-T}He^{T}$ with $T=T_1+T_2$ in the full space of determinants, projected on the reference,
   the singles and the doubles;
2. **the diagrams**: the 3 energy diagrams, the 14 diagrams of the singles equation and the 31 of the doubles
   equation (chapter 10, section 10.14; `BookPrograms/chapter10/ccsd_diagrams.py`);
3. **the $T_1$-dressed Hamiltonian**: $\tilde H=e^{-T_1}He^{T_1}$ is again a one- plus two-body operator, with
   integrals $\tilde h=XhY^{T}$, $\tilde v=XXvY^{T}Y^{T}$, $X=1-t_1$, $Y=1+t_1^{T}$, and CCSD is CCD with
   $\tilde H$ plus a four-term singles equation.

The third route is then turned into a forty-line CCSD solver and applied to the pairing model with a pair-breaking
particle-hole term, where the singles matter.

In [1]:
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True, linewidth=120)
es = np.einsum

## 1. The tools of week 47

The `CCD` class (the ten terms of the doubles equation) and the brute-force `DeterminantSpace` are those of
`week47.ipynb`; the cluster operator now also takes singles.

In [2]:
class CCD:
    """Coupled-cluster doubles for a Fock matrix f and antisymmetrised v = <pq||rs>."""

    names = ("drive", "fock_p", "fock_h", "pp", "hh", "ring", "Qa", "Qb", "Qc", "Qd")

    def __init__(self, f, v, n_occ):
        n = f.shape[0]
        o, u = slice(0, n_occ), slice(n_occ, n)
        self.no, self.nv = n_occ, n - n_occ
        self.foo, self.fvv = f[o, o], f[u, u]
        self.vvvoo = v[u, u, o, o]              # <ab||ij>
        self.vvvvv = v[u, u, u, u]              # <ab||cd>
        self.voooo = v[o, o, o, o]              # <kl||ij>
        self.vovvo = v[o, u, u, o]              # <kb||cj>
        self.voovv = v[o, o, u, u]              # <kl||cd>
        eo, ev = np.diag(self.foo), np.diag(self.fvv)
        self.D = (eo[:, None, None, None] + eo[None, :, None, None]
                  - ev[None, None, :, None] - ev[None, None, None, :])

    # ----- permutation operators acting on X[i, j, a, b]
    @staticmethod
    def Pij(x):
        return x - x.transpose(1, 0, 2, 3)

    @staticmethod
    def Pab(x):
        return x - x.transpose(0, 1, 3, 2)

    # ----- the ten terms
    def terms(self, t):
        es, Pij, Pab = np.einsum, self.Pij, self.Pab
        w = self.voovv
        return {
            "drive":  self.vvvoo.transpose(2, 3, 0, 1).copy(),
            "fock_p": Pab(es("bc,ijac->ijab", self.fvv, t)),
            "fock_h": -Pij(es("kj,ikab->ijab", self.foo, t)),
            "pp":     0.5 * es("abcd,ijcd->ijab", self.vvvvv, t),
            "hh":     0.5 * es("klij,klab->ijab", self.voooo, t),
            "ring":   Pij(Pab(es("kbcj,ikac->ijab", self.vovvo, t))),
            "Qa":     0.25 * es("klcd,ijcd,klab->ijab", w, t, t, optimize=True),
            "Qb":     0.5 * Pij(Pab(es("klcd,ikac,jlbd->ijab", w, t, t, optimize=True))),
            "Qc":     -0.5 * Pab(es("klcd,ijac,klbd->ijab", w, t, t, optimize=True)),
            "Qd":     -0.5 * Pij(es("klcd,ikab,jlcd->ijab", w, t, t, optimize=True)),
        }

    def residual(self, t, keep=None):
        """<Phi_ij^ab| Hbar |Phi_0>; `keep` selects a subset of the terms."""
        terms = self.terms(t)
        return sum(terms[k] for k in (keep or self.names))

    def energy(self, t):
        """Correlation energy 1/4 sum <ij||ab> t_ij^ab."""
        return 0.25 * float(np.einsum("ijab,ijab->", self.voovv, t))

    def first_order(self):
        """The MBPT amplitude <ab||ij>/D: the starting point of the iteration."""
        return self.vvvoo.transpose(2, 3, 0, 1) / self.D

    def antisymmetrise(self, t):
        """Project on the part antisymmetric in (i, j) and in (a, b): the only part T_2 depends on."""
        return 0.25 * self.Pij(self.Pab(t))

    def solve(self, mix=1.0, tol=1e-12, max_iter=2000, keep=None, verbose=False, project=True):
        """Damped Jacobi iteration t <- t + mix * R(t)/D, stopped on max|R|.

        With project=True the amplitudes are antisymmetrised after every step, which removes
        the rounding noise that would otherwise accumulate in the unphysical components."""
        t = self.first_order()
        history = []
        for it in range(1, max_iter + 1):
            R = self.residual(t, keep)
            history.append((self.energy(t), abs(R).max()))
            if verbose:
                print(f"{it:4d}  E_corr = {history[-1][0]: .12f}   max|R| = {history[-1][1]:.2e}")
            if history[-1][1] < tol:
                break
            if not np.isfinite(history[-1][1]) or history[-1][1] > 1e6:
                break
            t = t + mix * R / self.D
            if project:
                t = self.antisymmetrise(t)
        self.history, self.iterations = history, it
        self.converged = history[-1][1] < tol
        return t


class DeterminantSpace:
    """Brute-force second quantisation on bit strings, for validation only."""

    def __init__(self, n, n_particles):
        self.n, self.N = n, n_particles
        self.dets = [sum(1 << p for p in c) for c in itertools.combinations(range(n), n_particles)]
        self.index = {d: k for k, d in enumerate(self.dets)}
        self.ref = self.index[sum(1 << p for p in range(n_particles))]

    @staticmethod
    def _apply(ops, state):
        """ops = [('c', p), ('a', q), ...] applied right to left; returns (sign, state)."""
        sign = 1
        for kind, p in reversed(ops):
            occupied = (state >> p) & 1
            if (kind == 'a') != bool(occupied):
                return 0, None
            if bin(state & ((1 << p) - 1)).count("1") & 1:
                sign = -sign
            state ^= 1 << p
        return sign, state

    def hamiltonian(self, h, v):
        dim, n = len(self.dets), self.n
        H = np.zeros((dim, dim))
        nz1 = [(p, q) for p in range(n) for q in range(n) if h[p, q] != 0]
        nz2 = [idx for idx in itertools.product(range(n), repeat=4) if v[idx] != 0]
        for k, d in enumerate(self.dets):
            for p, q in nz1:
                s, e = self._apply([('c', p), ('a', q)], d)
                if s:
                    H[self.index[e], k] += s * h[p, q]
            for p, q, r, s_ in nz2:
                s, e = self._apply([('c', p), ('c', q), ('a', s_), ('a', r)], d)
                if s:
                    H[self.index[e], k] += 0.25 * s * v[p, q, r, s_]
        return H

    def cluster_operator(self, t, t1=None):
        """Matrix of T_2 = 1/4 sum t_ij^ab a+_a a+_b a_j a_i  (plus T_1 = sum t_i^a a+_a a_i)."""
        dim, N = len(self.dets), self.N
        T = np.zeros((dim, dim))
        no, nv = t.shape[0], t.shape[2]
        for k, d in enumerate(self.dets):
            if t1 is not None:
                for i, a in itertools.product(range(no), range(nv)):
                    s, e = self._apply([('c', N + a), ('a', i)], d)
                    if s:
                        T[self.index[e], k] += s * t1[i, a]
            for i, j, a, b in itertools.product(range(no), range(no), range(nv), range(nv)):
                s, e = self._apply([('c', N + a), ('c', N + b), ('a', j), ('a', i)], d)
                if s:
                    T[self.index[e], k] += 0.25 * s * t[i, j, a, b]
        return T

    def project_doubles(self, M, no, nv):
        """<Phi_ij^ab| M |Phi_0> as an array [i, j, a, b]."""
        R, N = np.zeros((no, no, nv, nv)), self.N
        for i, j, a, b in itertools.product(range(no), range(no), range(nv), range(nv)):
            s, e = self._apply([('c', N + a), ('c', N + b), ('a', j), ('a', i)], self.dets[self.ref])
            if s:
                R[i, j, a, b] = s * M[self.index[e], self.ref]
        return R

    def project_singles(self, M, no, nv):
        """<Phi_i^a| M |Phi_0> as an array [i, a]."""
        R, N = np.zeros((no, nv)), self.N
        for i, a in itertools.product(range(no), range(nv)):
            s, e = self._apply([('c', N + a), ('a', i)], self.dets[self.ref])
            R[i, a] = s * M[self.index[e], self.ref]
        return R


def expm_nilpotent(T):
    out, term = np.eye(len(T)), np.eye(len(T))
    for k in range(1, 20):
        term = term @ T / k
        out += term
        if abs(term).max() < 1e-15:
            break
    return out

## 2. A random problem and the brute-force answer

In [3]:
n, n_occ = 8, 4
nv = n - n_occ
rng = np.random.default_rng(48)
h = rng.normal(size=(n, n)); h = 0.5 * (h + h.T)
v = rng.normal(size=(n,) * 4)
v = v - v.transpose(1, 0, 2, 3); v = v - v.transpose(0, 1, 3, 2); v = 0.5 * (v + v.transpose(2, 3, 0, 1))
t1 = 0.3 * rng.normal(size=(n_occ, nv))
t2 = 0.3 * rng.normal(size=(n_occ, n_occ, nv, nv))
t2 = t2 - t2.transpose(1, 0, 2, 3); t2 = t2 - t2.transpose(0, 1, 3, 2)

def fock_matrix(h, v, n_occ):
    return h + sum(v[:, i, :, i] for i in range(n_occ))

def reference_energy(h, v, n_occ):
    occ = range(n_occ)
    return sum(h[i, i] for i in occ) + 0.5 * sum(v[i, j, i, j] for i in occ for j in occ)

space = DeterminantSpace(n, n_occ)
H = space.hamiltonian(h, v)
T = space.cluster_operator(t2, t1)
Hbar = expm_nilpotent(-T) @ H @ expm_nilpotent(T)
E_brute = Hbar[space.ref, space.ref] - H[space.ref, space.ref]
R1_brute = space.project_singles(Hbar, n_occ, nv)
R2_brute = space.project_doubles(Hbar, n_occ, nv)
print("brute force:  Delta E =", E_brute, "  max|R1| =", abs(R1_brute).max(), "  max|R2| =", abs(R2_brute).max())

brute force:  Delta E = -0.5228281907097836   max|R1| = 13.897890314668851   max|R2| = 29.726734357544846


## 3. The diagrams: 3 + 14 + 31 terms

In the order of the figures of chapter 10 (and of the slides). `W('o','v','v','o')` is the block
$\langle ma\Vert ei\rangle$, and so on.

In [4]:
def ccsd_diagram_terms(f, v, n_occ, t1, t2):
    """The diagram sums: (energy terms, 14 singles terms, 31 doubles terms)."""
    n = f.shape[0]
    o, u = list(range(n_occ)), list(range(n_occ, n))
    foo, fvv, fov, fvo = f[np.ix_(o, o)], f[np.ix_(u, u)], f[np.ix_(o, u)], f[np.ix_(u, o)]
    def W(*blk): return v[np.ix_(*[o if c == 'o' else u for c in blk])]
    def Pij(x): return x - x.transpose(1, 0, 2, 3)
    def Pab(x): return x - x.transpose(0, 1, 3, 2)
    oovv = W('o', 'o', 'v', 'v')
    E = [es("ia,ia->", fov, t1),
         0.25 * es("ijab,ijab->", oovv, t2),
         0.5 * es("ijab,ia,jb->", oovv, t1, t1)]
    S = [fvo.T,                                                       #  1  f_ai
         es("ae,ie->ia", fvv, t1),                                    #  2  f_ae t_i^e
         -es("mi,ma->ia", foo, t1),                                   #  3 -f_mi t_m^a
         es("maei,me->ia", W('o', 'v', 'v', 'o'), t1),                #  4  <ma||ei> t_m^e
         es("me,imae->ia", fov, t2),                                  #  5  f_me t_im^ae
         0.5 * es("amef,imef->ia", W('v', 'o', 'v', 'v'), t2),        #  6  1/2 <am||ef> t_im^ef
         -0.5 * es("mnei,mnea->ia", W('o', 'o', 'v', 'o'), t2),       #  7 -1/2 <mn||ei> t_mn^ea
         -es("me,ie,ma->ia", fov, t1, t1),                            #  8
         es("amef,ie,mf->ia", W('v', 'o', 'v', 'v'), t1, t1),         #  9
         -es("mnei,me,na->ia", W('o', 'o', 'v', 'o'), t1, t1),        # 10
         es("mnef,me,nifa->ia", oovv, t1, t2),                        # 11
         -0.5 * es("mnef,ie,mnaf->ia", oovv, t1, t2),                 # 12
         -0.5 * es("mnef,na,mief->ia", oovv, t1, t2),                 # 13
         -es("mnef,ie,ma,nf->ia", oovv, t1, t1, t1)]                  # 14
    D = [W('v', 'v', 'o', 'o').transpose(2, 3, 0, 1),                                 #  1
         Pij(es("abej,ie->ijab", W('v', 'v', 'v', 'o'), t1)),                        #  2
         -Pab(es("amij,mb->ijab", W('v', 'o', 'o', 'o'), t1)),                       #  3
         Pab(es("be,ijae->ijab", fvv, t2)),                                          #  4
         -Pij(es("mi,mjab->ijab", foo, t2)),                                         #  5
         0.5 * es("abef,ijef->ijab", W('v', 'v', 'v', 'v'), t2),                     #  6
         0.5 * es("mnij,mnab->ijab", W('o', 'o', 'o', 'o'), t2),                     #  7
         Pij(Pab(es("mbej,imae->ijab", W('o', 'v', 'v', 'o'), t2))),                 #  8
         0.5 * Pij(es("abef,ie,jf->ijab", W('v', 'v', 'v', 'v'), t1, t1)),           #  9
         0.5 * Pab(es("mnij,ma,nb->ijab", W('o', 'o', 'o', 'o'), t1, t1)),           # 10
         -Pij(Pab(es("mbej,ie,ma->ijab", W('o', 'v', 'v', 'o'), t1, t1))),           # 11
         0.25 * es("mnef,ijef,mnab->ijab", oovv, t2, t2),                            # 12
         0.5 * Pij(Pab(es("mnef,imae,njfb->ijab", oovv, t2, t2))),                   # 13
         -0.5 * Pab(es("mnef,ijae,mnbf->ijab", oovv, t2, t2)),                       # 14
         -0.5 * Pij(es("mnef,mief,njab->ijab", oovv, t2, t2)),                       # 15
         -Pij(es("me,ie,mjab->ijab", fov, t1, t2)),                                  # 16
         -Pab(es("me,ijae,mb->ijab", fov, t2, t1)),                                  # 17
         Pij(Pab(es("amef,ie,mjfb->ijab", W('v', 'o', 'v', 'v'), t1, t2))),          # 18
         -0.5 * Pab(es("amef,ijef,mb->ijab", W('v', 'o', 'v', 'v'), t2, t1)),        # 19
         Pab(es("bmef,ijae,mf->ijab", W('v', 'o', 'v', 'v'), t2, t1)),               # 20
         -Pij(Pab(es("mnej,imae,nb->ijab", W('o', 'o', 'v', 'o'), t2, t1))),         # 21
         0.5 * Pij(es("mnej,ie,mnab->ijab", W('o', 'o', 'v', 'o'), t1, t2)),         # 22
         -Pij(es("mnei,me,njab->ijab", W('o', 'o', 'v', 'o'), t1, t2)),              # 23
         -0.5 * Pij(Pab(es("amef,ie,jf,mb->ijab", W('v', 'o', 'v', 'v'), t1, t1, t1))),   # 24
         0.5 * Pij(Pab(es("mnej,ie,ma,nb->ijab", W('o', 'o', 'v', 'o'), t1, t1, t1))),    # 25
         0.25 * Pij(es("mnef,ie,mnab,jf->ijab", oovv, t1, t2, t1)),                  # 26
         -Pij(Pab(es("mnef,ie,ma,njfb->ijab", oovv, t1, t1, t2))),                   # 27
         0.25 * Pab(es("mnef,ma,ijef,nb->ijab", oovv, t1, t2, t1)),                  # 28
         -Pij(es("mnef,me,if,njab->ijab", oovv, t1, t1, t2)),                        # 29
         -Pab(es("mnef,ijae,mb,nf->ijab", oovv, t2, t1, t1)),                        # 30
         0.25 * Pij(Pab(es("mnef,ie,ma,jf,nb->ijab", oovv, t1, t1, t1, t1)))]        # 31
    return E, S, D


f = fock_matrix(h, v, n_occ)
E_terms, S_terms, D_terms = ccsd_diagram_terms(f, v, n_occ, t1, t2)
print("energy :  3 diagrams, difference from brute force", f"{abs(sum(E_terms) - E_brute):.1e}")
print("singles: 14 diagrams, largest difference        ", f"{abs(sum(S_terms) - R1_brute).max():.1e}")
print("doubles: 31 diagrams, largest difference        ", f"{abs(sum(D_terms) - R2_brute).max():.1e}")
by_degree = {"linear (1, 4-8)": [0, 3, 4, 5, 6, 7], "T1 linear (2, 3)": [1, 2], "quadratic (9-23)": list(range(8, 23)),
             "cubic (24-30)": list(range(23, 30)), "quartic (31)": [30]}
print("\nsize of the groups of the doubles equation (largest element):")
for label, idx in by_degree.items():
    print(f"  {label:20s} {abs(sum(D_terms[k] for k in idx)).max():10.4f}")

energy :  3 diagrams, difference from brute force 2.6e-15
singles: 14 diagrams, largest difference         6.2e-15
doubles: 31 diagrams, largest difference         1.1e-14

size of the groups of the doubles equation (largest element):
  linear (1, 4-8)         21.5080
  T1 linear (2, 3)         3.6396
  quadratic (9-23)        16.1967
  cubic (24-30)            5.5521
  quartic (31)             0.8153


### The signs from the topology of the diagrams

Each diagram is specified by its $H_N$ fragment -- `('f', out, in)` or `('v', (out, in), (out, in))`, one pair per
vertex -- and by its bars, each a list of (hole, particle) vertices. The sign rule $(-1)^{h+l}$ needs the number of
hole lines $h$ and the number of loops $l$, with the external lines closed through the bra ($i$ with $a$, $j$ with
$b$). Counting loops is counting the connected components of the graph whose nodes are the vertices and whose
edges are the lines. The signs of all 48 diagrams come out as in the equations above.

In [5]:
HOLES = set("ijklmn")
E_SPEC = [
 (('f','i','a'), [[('i','a')]]),
 (('v',('i','a'),('j','b')), [[('i','a'),('j','b')]]),
 (('v',('i','a'),('j','b')), [[('i','a')],[('j','b')]]),
]
S_SPEC = [
 (('f','a','i'), []),
 (('f','a','e'), [[('i','e')]]),
 (('f','m','i'), [[('m','a')]]),
 (('v',('m','e'),('a','i')), [[('m','e')]]),
 (('f','m','e'), [[('i','a'),('m','e')]]),
 (('v',('a','e'),('m','f')), [[('i','e'),('m','f')]]),
 (('v',('m','e'),('n','i')), [[('m','e'),('n','a')]]),
 (('f','m','e'), [[('i','e')],[('m','a')]]),
 (('v',('a','e'),('m','f')), [[('i','e')],[('m','f')]]),
 (('v',('m','e'),('n','i')), [[('m','e')],[('n','a')]]),
 (('v',('m','e'),('n','f')), [[('m','e')],[('n','f'),('i','a')]]),
 (('v',('m','e'),('n','f')), [[('i','e')],[('m','a'),('n','f')]]),
 (('v',('m','e'),('n','f')), [[('n','a')],[('m','e'),('i','f')]]),
 (('v',('m','e'),('n','f')), [[('i','e')],[('m','a')],[('n','f')]]),
]
D_SPEC = [
 (('v',('a','i'),('b','j')), []),
 (('v',('a','e'),('b','j')), [[('i','e')]]),
 (('v',('a','i'),('m','j')), [[('m','b')]]),
 (('f','b','e'), [[('i','a'),('j','e')]]),
 (('f','m','i'), [[('m','a'),('j','b')]]),
 (('v',('a','e'),('b','f')), [[('i','e'),('j','f')]]),
 (('v',('m','i'),('n','j')), [[('m','a'),('n','b')]]),
 (('v',('m','e'),('b','j')), [[('i','a'),('m','e')]]),
 (('v',('a','e'),('b','f')), [[('i','e')],[('j','f')]]),
 (('v',('m','i'),('n','j')), [[('m','a')],[('n','b')]]),
 (('v',('m','e'),('b','j')), [[('i','e')],[('m','a')]]),
 (('v',('m','e'),('n','f')), [[('i','e'),('j','f')],[('m','a'),('n','b')]]),
 (('v',('m','e'),('n','f')), [[('i','a'),('m','e')],[('n','f'),('j','b')]]),
 (('v',('m','e'),('n','f')), [[('i','a'),('j','e')],[('m','b'),('n','f')]]),
 (('v',('m','e'),('n','f')), [[('m','e'),('i','f')],[('n','a'),('j','b')]]),
 (('f','m','e'), [[('i','e')],[('m','a'),('j','b')]]),
 (('f','m','e'), [[('i','a'),('j','e')],[('m','b')]]),
 (('v',('a','e'),('m','f')), [[('i','e')],[('m','f'),('j','b')]]),
 (('v',('a','e'),('m','f')), [[('i','e'),('j','f')],[('m','b')]]),
 (('v',('b','e'),('m','f')), [[('i','a'),('j','e')],[('m','f')]]),
 (('v',('m','e'),('n','j')), [[('i','a'),('m','e')],[('n','b')]]),
 (('v',('m','e'),('n','j')), [[('i','e')],[('m','a'),('n','b')]]),
 (('v',('m','e'),('n','i')), [[('m','e')],[('n','a'),('j','b')]]),
 (('v',('a','e'),('m','f')), [[('i','e')],[('j','f')],[('m','b')]]),
 (('v',('m','e'),('n','j')), [[('i','e')],[('m','a')],[('n','b')]]),
 (('v',('m','e'),('n','f')), [[('i','e')],[('m','a'),('n','b')],[('j','f')]]),
 (('v',('m','e'),('n','f')), [[('i','e')],[('m','a')],[('n','f'),('j','b')]]),
 (('v',('m','e'),('n','f')), [[('m','a')],[('i','e'),('j','f')],[('n','b')]]),
 (('v',('m','e'),('n','f')), [[('m','e')],[('i','f')],[('n','a'),('j','b')]]),
 (('v',('m','e'),('n','f')), [[('i','a'),('j','e')],[('m','b')],[('n','f')]]),
 (('v',('m','e'),('n','f')), [[('i','e')],[('m','a')],[('j','f')],[('n','b')]]),
]


def holes_and_loops(fragment, bars, external=(("i", "a"), ("j", "b"))):
    vertices = [(fragment[1], fragment[2])] if fragment[0] == "f" else [fragment[1], fragment[2]]
    for bar in bars:
        vertices += list(bar)
    labels = {l for v in vertices for l in v}
    vertices += [p for p in external if p[0] in labels and p[1] in labels]      # the bra closes the open lines
    labels = {l for v in vertices for l in v}
    touching = {l: [k for k, v in enumerate(vertices) if l in v] for l in labels}
    seen, loops = set(), 0
    for start in range(len(vertices)):
        if start in seen:
            continue
        loops += 1
        stack = [start]
        while stack:
            k = stack.pop()
            if k in seen:
                continue
            seen.add(k)
            for l in vertices[k]:
                stack += [j for j in touching[l] if j not in seen]
    return sum(1 for l in labels if l in HOLES), loops


signs = {"E": "+ + +", "T1": "+ + - + + + - - + - + - - -",
         "T2": "+ + - + - + + + + + - + + - - - - + - + - + - - + + - + - - +"}
for name, terms, ext in (("E", E_SPEC, ()), ("T1", S_SPEC, (("i", "a"),)), ("T2", D_SPEC, (("i", "a"), ("j", "b")))):
    rule = []
    for fragment, bars in terms:
        hl = holes_and_loops(fragment, bars, ext)
        rule.append("+" if sum(hl) % 2 == 0 else "-")
    print(f"{name:3s} {len(terms):2d} diagrams, signs from (-1)^(h+l) agree with the equations: {' '.join(rule) == signs[name]}")
print("examples (h, l):  singles (3)", holes_and_loops(*S_SPEC[2], (("i", "a"),)),
      "  singles (7)", holes_and_loops(*S_SPEC[6], (("i", "a"),)),
      "  doubles (8)", holes_and_loops(*D_SPEC[7]), "  doubles (11)", holes_and_loops(*D_SPEC[10]))

E    3 diagrams, signs from (-1)^(h+l) agree with the equations: True
T1  14 diagrams, signs from (-1)^(h+l) agree with the equations: True
T2  31 diagrams, signs from (-1)^(h+l) agree with the equations: True
examples (h, l):  singles (3) (2, 1)   singles (7) (3, 2)   doubles (8) (3, 3)   doubles (11) (3, 2)


## 4. The $T_1$-dressed Hamiltonian

$e^{-T_1}a_p^\dagger e^{T_1}=\sum_q a_q^\dagger X_{qp}$ and $e^{-T_1}a_pe^{T_1}=\sum_q a_qY_{qp}$ with
$X=1-t_1$ (the block $X_{ai}=-t_i^a$) and $Y=1+t_1^{T}$ (the block $Y_{ia}=+t_i^a$). Hence
$\tilde h_{pq}=\sum X_{pp'}h_{p'q'}Y_{qq'}$, $\tilde v_{pqrs}=\sum X_{pp'}X_{qq'}v_{p'q'r's'}Y_{rr'}Y_{ss'}$,
and since $[T_1,T_2]=0$,

$$e^{-T}He^{T}=e^{-T_2}\tilde He^{T_2}.$$

CCSD is CCD for $\tilde H$ (not Hermitian, but still a one- plus two-body operator), plus the singles projection,
which has only four terms because one $T_2$ and a two-body operator can leave a single in three ways:

$$0=\tilde f_{ai}+\tilde f_{kc}t_{ik}^{ac}+\tfrac12\langle ak\Vert cd\rangle^{\sim}t_{ik}^{cd}
  -\tfrac12\langle kl\Vert ic\rangle^{\sim}t_{kl}^{ac},\qquad
  \Delta E=\tilde E_{\rm ref}-E_{\rm ref}+\tfrac14\langle kl\Vert cd\rangle^{\sim}t_{kl}^{cd}.$$

In [6]:
def dress(h, v, n_occ, t1):
    """Integrals of exp(-T1) H exp(T1)."""
    n = h.shape[0]
    X, Y = np.eye(n), np.eye(n)
    X[n_occ:, :n_occ] = -t1.T            # X[a, i] = -t_i^a
    Y[:n_occ, n_occ:] = t1               # Y[i, a] = +t_i^a
    ht = X @ h @ Y.T
    vt = es("pP,qQ,PQRS,rR,sS->pqrs", X, X, v, Y, Y, optimize=True)
    return ht, vt


def ccsd_residuals(h, v, n_occ, t1, t2):
    """Energy, singles and doubles residuals of CCSD from the T1-dressed Hamiltonian."""
    ht, vt = dress(h, v, n_occ, t1)
    ft = fock_matrix(ht, vt, n_occ)
    o, u = slice(0, n_occ), slice(n_occ, h.shape[0])
    ccd = CCD(ft, vt, n_occ)
    energy = reference_energy(ht, vt, n_occ) - reference_energy(h, v, n_occ) + ccd.energy(t2)
    R1 = (ft[u, o].T                                             # f~_ai
          + es("kc,ikac->ia", ft[o, u], t2)                      # f~_kc t_ik^ac
          + 0.5 * es("akcd,ikcd->ia", vt[u, o, u, u], t2)        # 1/2 <ak||cd>~ t_ik^cd
          - 0.5 * es("klic,klac->ia", vt[o, o, o, u], t2))       # -1/2 <kl||ic>~ t_kl^ac
    R2 = ccd.residual(t2)                                        # the ten CCD terms with dressed integrals
    return energy, R1, R2


E_dressed, R1_dressed, R2_dressed = ccsd_residuals(h, v, n_occ, t1, t2)
print("energy :  dressed - brute force", f"{abs(E_dressed - E_brute):.1e}")
print("singles:  largest difference   ", f"{abs(R1_dressed - R1_brute).max():.1e}")
print("doubles:  largest difference   ", f"{abs(R2_dressed - R2_brute).max():.1e}")
ht, vt = dress(h, v, n_occ, t1)
print("dressed Hamiltonian Hermitian? ", np.allclose(ht, ht.T), np.allclose(vt, vt.transpose(2, 3, 0, 1)))
Hd = space.hamiltonian(ht, vt)
T1 = space.cluster_operator(0 * t2, t1)
print("matrix of the dressed integrals = exp(-T1) H exp(T1):",
      f"{abs(Hd - expm_nilpotent(-T1) @ H @ expm_nilpotent(T1)).max():.1e}")

energy :  dressed - brute force 7.8e-16
singles:  largest difference    5.3e-15
doubles:  largest difference    1.4e-14
dressed Hamiltonian Hermitian?  False False
matrix of the dressed integrals = exp(-T1) H exp(T1): 7.1e-15


### Which diagrams does each dressed vertex contain?

Expanding the dressed integrals in powers of $t_1$ sorts the 14 + 31 diagrams into the four singles terms and the
ten CCD terms:

In [7]:
ccd_dressed = CCD(fock_matrix(ht, vt, n_occ), vt, n_occ)
dressed_terms = ccd_dressed.terms(t2)
ft = fock_matrix(ht, vt, n_occ)
o, u = slice(0, n_occ), slice(n_occ, n)
groups_doubles = {
    "drive":           ("driving term",    [1, 2, 3, 9, 10, 11, 24, 25, 31]),
    "fock_p + fock_h": ("Fock terms",      [4, 5, 16, 17, 20, 23, 29, 30]),
    "pp":              ("pp ladder",       [6, 19, 28]),
    "hh":              ("hh ladder",       [7, 22, 26]),
    "ring":            ("ring",            [8, 18, 21, 27]),
    "Qa + Qb + Qc + Qd": ("quadratic",     [12, 13, 14, 15]),
}
for key, (label, numbers) in groups_doubles.items():
    ours = sum(dressed_terms[k.strip()] for k in key.split("+"))
    diagrams = sum(D_terms[k - 1] for k in numbers)
    print(f"doubles, dressed {label:13s} = diagrams {str(numbers):38s} difference {abs(ours - diagrams).max():.1e}")
groups_singles = {
    "f~_ai":                  (ft[u, o].T,                                        [1, 2, 3, 4, 8, 9, 10, 14]),
    "f~_kc t_ik^ac":          (es("kc,ikac->ia", ft[o, u], t2),                   [5, 11]),
    "1/2 <ak||cd>~ t_ik^cd":  (0.5 * es("akcd,ikcd->ia", vt[u, o, u, u], t2),     [6, 13]),
    "-1/2 <kl||ic>~ t_kl^ac": (-0.5 * es("klic,klac->ia", vt[o, o, o, u], t2),    [7, 12]),
}
for label, (ours, numbers) in groups_singles.items():
    diagrams = sum(S_terms[k - 1] for k in numbers)
    print(f"singles, {label:23s} = diagrams {str(numbers):28s} difference {abs(ours - diagrams).max():.1e}")

doubles, dressed driving term  = diagrams [1, 2, 3, 9, 10, 11, 24, 25, 31]       difference 1.8e-15
doubles, dressed Fock terms    = diagrams [4, 5, 16, 17, 20, 23, 29, 30]         difference 4.2e-15
doubles, dressed pp ladder     = diagrams [6, 19, 28]                            difference 1.8e-15
doubles, dressed hh ladder     = diagrams [7, 22, 26]                            difference 1.8e-15
doubles, dressed ring          = diagrams [8, 18, 21, 27]                        difference 5.3e-15
doubles, dressed quadratic     = diagrams [12, 13, 14, 15]                       difference 4.9e-15
singles, f~_ai                   = diagrams [1, 2, 3, 4, 8, 9, 10, 14]   difference 3.6e-15
singles, f~_kc t_ik^ac           = diagrams [5, 11]                      difference 5.8e-15
singles, 1/2 <ak||cd>~ t_ik^cd   = diagrams [6, 13]                      difference 2.7e-15
singles, -1/2 <kl||ic>~ t_kl^ac  = diagrams [7, 12]                      difference 2.2e-15


Three routes, one answer: the 45 diagrams with $T_1$ are the 10 diagrams of CCD (plus four for the singles) drawn
with dressed vertices.

## 5. A CCSD solver, and a model in which singles matter

The pairing model with a particle-hole term that breaks pairs (chapter 10, section 10.10),

$$V=-\frac g2\sum_{pq}a^\dagger_{p+}a^\dagger_{p-}a_{q-}a_{q+}
    -\frac f2\sum_{pqr}\bigl(a^\dagger_{p+}a^\dagger_{p-}a_{q-}a_{r+}+{\rm h.c.}\bigr).$$

In [8]:
class PairingPHModel:
    """Pairing plus particle-hole (pair-breaking) interaction, in spin orbitals."""

    def __init__(self, levels=4, particles=4, g=1.0, f=0.0, xi=1.0):
        self.n, self.n_occ = 2 * levels, particles
        self.h = np.diag([xi * (p // 2) for p in range(self.n)]).astype(float)
        self.v = np.zeros((self.n,) * 4)
        for p in range(levels):
            for q in range(levels):
                self._add(-0.5 * g, 2 * p, 2 * p + 1, 2 * q, 2 * q + 1)
                for r in range(levels):
                    self._add(-0.5 * f, 2 * p, 2 * p + 1, 2 * r, 2 * q + 1)
                    self._add(-0.5 * f, 2 * r, 2 * q + 1, 2 * p, 2 * p + 1)

    def _add(self, c, p, q, r, s):
        """c * a+_p a+_q a_s a_r into the antisymmetrised tensor."""
        self.v[p, q, r, s] += c; self.v[q, p, r, s] -= c
        self.v[p, q, s, r] -= c; self.v[q, p, s, r] += c

    def exact_energy(self):
        space = DeterminantSpace(self.n, self.n_occ)
        return np.linalg.eigvalsh(space.hamiltonian(self.h, self.v))[0]


class CCSD:
    """CCSD by Jacobi iteration on the residuals of the T1-dressed Hamiltonian."""

    def __init__(self, h, v, n_occ):
        self.h, self.v, self.n_occ = h, v, n_occ
        eps = np.diag(fock_matrix(h, v, n_occ))
        eo, ev = eps[:n_occ], eps[n_occ:]
        self.D1 = eo[:, None] - ev[None, :]
        self.D2 = (eo[:, None, None, None] + eo[None, :, None, None]
                   - ev[None, None, :, None] - ev[None, None, None, :])

    def solve(self, singles=True, tol=1e-11, max_iter=500, mix=1.0):
        no, nv = self.D1.shape
        t1, t2 = np.zeros((no, nv)), np.zeros((no, no, nv, nv))
        for it in range(1, max_iter + 1):
            energy, R1, R2 = ccsd_residuals(self.h, self.v, self.n_occ, t1, t2)
            if not singles:
                R1 = 0 * R1
            if max(abs(R1).max(), abs(R2).max()) < tol:
                break
            t1 = t1 + mix * R1 / self.D1
            t2 = t2 + mix * R2 / self.D2
            t2 = 0.25 * (t2 - t2.transpose(1, 0, 2, 3) - t2.transpose(0, 1, 3, 2) + t2.transpose(1, 0, 3, 2))
        self.iterations = it
        return energy, t1, t2


print(f"{'g':>5s} {'f':>6s} {'CCD':>12s} {'CCSD':>12s} {'exact':>12s} {'max|t1|':>9s} {'iter':>5s}")
for g, fph in ((0.5, 0.0), (0.5, 0.025), (0.7, 0.05), (1.0, 0.05), (0.5, 0.25)):
    model = PairingPHModel(4, 4, g, fph)
    e_ref = reference_energy(model.h, model.v, model.n_occ)
    solver = CCSD(model.h, model.v, model.n_occ)
    e_d, _, _ = solver.solve(singles=False, mix=0.7)
    e_sd, t1s, _ = solver.solve(mix=0.7)
    print(f"{g:5.2f} {fph:6.3f} {e_ref + e_d:12.7f} {e_ref + e_sd:12.7f} {model.exact_energy():12.7f} "
          f"{abs(t1s).max():9.1e} {solver.iterations:5d}")

    g      f          CCD         CCSD        exact   max|t1|  iter
 0.50  0.000    1.4166377    1.4166377    1.4167743   0.0e+00    33


 0.50  0.025    1.3476038    1.3469075    1.3471327   1.1e-02    35


 0.70  0.050    0.9707634    0.9681476    0.9697588   1.9e-02    40
 1.00  0.050    0.4447429    0.4423659    0.4505823   1.7e-02    44


 0.50  0.250    0.6033126    0.5467210    0.5511916   8.4e-02    44


At $f=0$ the singles vanish identically and CCSD is CCD; as soon as the interaction can break a pair the singles
appear, and at $g=0.5$, $f=0.25$ they move the energy from $0.603$ to $0.547$ against an exact $0.551$. These are
the numbers of the table of section 10.10 of the lecture notes, obtained there with the factored (intermediate)
form of the same equations.